In [1]:
import networkcommons as nc
import pandas as pd
import decoupler as dc
import corneto as cn
import numpy as np

(CORNETO) Jul 18 04:03:42 PM - WARNING : No backend found. You can install one of the supported backend by `pip install cvxpy` or `pip install picos`.


## TODOs:
- Fix backend install for CORNETO. Add CVXPY as dependency? Produces error when loading the module.
- Clean data description table: Right now, there is a conceptual mix of datasets (e.g. decryptm, panacea), with data used for tests (can remain there but it is conceptually different) and method-specific data. In order to make the data module clearer from a conceptual perspective, we need to separate data description from its link to the method. For instance, moon data is actually NCI60 dataset right? It would make sense to have a description of what this dataset contains and how it was calculated. I will provide some examples. Wondering whether we can even incorporate some high-level metadata (e.g. number of contrasts, observations, biological contexts). Wondering also if a dedicated preprocessing field makes sense. Can be discussed tomorrow.
- Ask aurelien to provide description and preprocessing information for the NCI60 dataset.
- Message to user when data is downloading. If connection is poor, this may not happen that fast.
- Single function to get PANACEA TF data. Wondering if it makes sense to do have a parallel dataframe with the TF level esitmates instead of performing the whole analysis. TBD.

## CONTRIBUTIONS GUIDELINES:
Ideally, a new contributor should provide, at least, the following information to include a dataset in the networkcommons:
- Description of the dataset
- Preprocessing steps
- Link to the dataset file (ideally a CSV file)
- Link to the dataset publication
- Link to the database source

Maybe we can draft an example of this

In [2]:
nc.data.omics.datasets()

{'decryptm': 'Drug perturbation proteomics and phosphoproteomics data',
 'panacea': 'Pancancer Analysis of Chemical Entity Activity RNA-Seq data',
 'test': 'Small RNA-Seq data for unit tests',
 'moon': 'Database files for running MOON',
 'cosmos': 'Database files for running COSMOS (MetaPKN)'}

In [3]:
data = {
    "id": ["panacea", "decryptm"],
    "description": [
        "This dataset profiles the response of 11 cell line to 32 kinase inhibitors using bulk transcriptomics. The data loader provides the count-level data for all the samples in the study together with sample information and drug-targets.",
        "Perturbational phosphoproteomics study where the response of N cell lines to X drugs is profiled in a dose and time-dependent manner. Not all the cell-line / dose / time combinations were explored. Authors use a custom strategy to fit dose and time response curves to peptide-level data. Our data loaders provide a list of all the curve files and their associated experiments. Users can then download specific experiments or all experiments and use protein-level features as input for the network methods."
    ],
    "publication_url": ["https://doi.org/10.1016/j.xcrm.2021.100492", "https://doi.org/10.1126/science.ade3925"],
    "database_url": ["https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE186341", "https://www.ebi.ac.uk/pride/archive/projects/PXD037285"]
}

df = pd.DataFrame(data)
pd.set_option('display.max_colwidth', None)
df

,id,description,publication_url,database_url
0,panacea,This dataset profiles the response of 11 cell line to 32 kinase inhibitors using bulk transcriptomics. The data loader provides the count-level data for all the samples in the study together with sample information and drug-targets.,https://doi.org/10.1016/j.xcrm.2021.100492,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE186341
1,decryptm,Perturbational phosphoproteomics study where the response of N cell lines to X drugs is profiled in a dose and time-dependent manner. Not all the cell-line / dose / time combinations were explored. Authors use a custom strategy to fit dose and time response curves to peptide-level data. Our data loaders provide a list of all the curve files and their associated experiments. Users can then download specific experiments or all experiments and use protein-level features as input for the network methods.,https://doi.org/10.1126/science.ade3925,https://www.ebi.ac.uk/pride/archive/projects/PXD037285


In [ ]:
panacea_countdata, panacea_metadata = nc.data.omics.panacea()

## Thinking whether all of the following shall be part of the package or not. 
## Ideally we want to reduce users' overhead. But as discussed, I like the 
## capacity of doing DE analysis within the NC. I would provide preprocessed data here
singlec_metadata = panacea_metadata[(panacea_metadata['group'] == 'ASPC_DMSO') | (panacea_metadata['group'] == 'ASPC_AFATINIB')]
singlec_samples = singlec_metadata['sample_ID'].tolist()
singlec_countdata = panacea_countdata[['gene_symbol'] + singlec_samples]
results = nc.data.omics.deseq2(singlec_countdata, singlec_metadata, test_group="ASPC_AFATINIB", ref_group="ASPC_DMSO")
decoupler_input = nc._utils.decoupler_formatter(results, 'stat')
collectri_net = dc.get_collectri()
dc_estimates, dc_pvals = dc.run_ulm(decoupler_input, collectri_net)
measurements = nc.utils.targetlayer_formatter(dc_estimates.T)

In [4]:
# ideally
# repeat cell_1, cell_2, cell_3 900 times
def load_panacea_tf():
    cells = np.repeat(['cell_1', 'cell_2', 'cell_3'], 300)
    drugs = np.tile(['drug_1', 'drug_2', 'drug_3'], 300)
    drug_targets = np.tile(['protein_a;protein_b', 'protein_c;protein_d', 'protein_e;protein_f'], 300)
    tfs = np.tile(['tf_' + str(i) for i in range(100)], 9)
    score = np.random.normal(0, 1, 900)
    df = pd.DataFrame({'cell': cells, 'drug': drugs, 'tf': tfs, 'score': score, 'drug_targets': drug_targets})
    return df

panacea_df = load_panacea_tf()
panacea_df.head()

,cell,drug,tf,score,drug_targets
0,cell_1,drug_1,tf_0,-0.219960,protein_a;protein_b
1,cell_1,drug_2,tf_1,-0.175496,protein_c;protein_d
2,cell_1,drug_3,tf_2,0.406253,protein_e;protein_f
3,cell_1,drug_1,tf_3,-1.006180,protein_a;protein_b
4,cell_1,drug_2,tf_4,-0.516930,protein_c;protein_d


In [6]:
# import plotly
import plotly.express as px

def plot_panacea():
    panacea_toplot = panacea_df.copy()
    panacea_toplot['cell_drug'] = panacea_toplot['cell'] + '_' + panacea_toplot['drug']
    panacea_toplot = panacea_toplot.pivot(index='tf', columns='cell_drug', values='score')
    fig = px.imshow(panacea_toplot.T, color_continuous_scale='RdBu', title='Panacea TFs', labels={'x': 'TF', 'y': 'Cell_Drug'})
    # Update layout settings
    fig.update_layout(
        autosize=True,
        width=1000,
        height=300
    )
    return fig

# this should be something like nc.omics.panacea.plot()
plot_panacea()